# Diffusion Limited Aggregation 

#### Author : B. Militzer, University of California, Berkeley 
#### Date   : Sept. 26, 2018

#### Read "The Science of Fractal Images", Ed. Peitgen and Saupe, p. 37 (1988)


In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [3]:
%matplotlib osx

In [4]:
#note, this function expects a matrix A[ix,iy] 
#and then displays so that A[:,0] is the lowest row of pixels
def display(A):
    maxX = A.shape[0]
    maxY = A.shape[1]
    B = np.zeros((maxY, maxX))
    for ix in range(0,maxX):
        for iy in range(0,maxY):
            B[maxY-1-iy,ix] = A[ix,iy]

    #Display the graphics outside of the notebook. 
    #On a PC, use '%matplotlib qt' instead.
    %matplotlib osx 
    
    plt.rcParams['figure.figsize'] = [6, 6/maxX*maxY]
    plt.imshow(B); 
    plt.axis('off'); 
    plt.show()
    plt.draw()
    plt.pause(0.01)

In [5]:
nParticles = 25000
maxX = 500
maxY = 250

In [6]:
# Initialize matrix containing all 2D grid points A(x,y)
# 0 <= x < maxX
# 0 <= y < maxY
# A(x,y)=0 ... site is empty
# A(x,y)>0 ... site is filled
A = np.zeros((maxX, maxY))

# Introduce a sticky wall at the bottom 
# by filling the lowest row of pixels with particles
A[:,0] = 1
print(A.transpose())

[[1. 1. 1. ... 1. 1. 1.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [7]:
#test the display routine
#display(A)

In [8]:
# To save computer time, we want to inject the new particle not too far
# above growing aggregate. We inject at on a line 'yStart', which
# keeps being increased so that it is always 'yBuffer' lines above the
# highest structure
yBuffer = 5
yStart  = 1 + yBuffer

In [9]:
for i in range(0,nParticles):
    # Compute new starting point on the line y=yStart
    x  = np.random.randint(0,maxX)
    y  = yStart; #always start at upper limit

    while True:
        xOrg = x
        yOrg = y

        r = np.random.random(); # Random float:  0.0 <= r < 1.0
        #based on the value of 'r', move the particle
        #left, right, up, or down and change x and y accordingly
        if r <= 0.25:
            x = x - 1 

        elif r > 0.25 and r <= 0.5:
            y = y + 1
            
        elif r > 0.5 and r <= 0.75:
            x = x + 1
            
        else:
            y = y - 1
        
        #now apply periodic boundary conditions to 'x'
        if x == -1:
            x = maxX - 1
        if x == maxX:
            x = 0
        
        if (A[x,y] == 1 or y>yStart): 
            x = xOrg
            y = yOrg
            continue; # if this site has been taken try moving in a different direction
        
        #determine the x coordionates of the left and right neighbors
        #store them in 'xm' and 'xp' and apply periodic boundary conditions again
        xp = x + 1
        xm = x - 1
        yp = y + 1
        ym = y - 1
        if xm == -1:
            xm = maxX - 1

        if xp == maxX:
            xp = 0
 
        # Determine if any neighboring site is occupied
        # if that is the case, enter the following 'if' clause

        neighbor_is_filled = False
        if A[xp, y] == 1 or A[xm, y] == 1 or A[x,yp] == 1 or A[x,ym] == 1:
            neighbor_is_filled = True
        
        if neighbor_is_filled: 
            A[x,y] = 1
            if (y+yBuffer>yStart and y+yBuffer<maxY): 
                yStart = y+yBuffer

            if (i%1000==0): 
                print(f'i= {i} \tx={x} \ty={y} \tyStart={yStart}')

            nNewParticlesPerFrame = 1000 
            if (i%nNewParticlesPerFrame==0): 
                display(A)
                
            break # particle was attached, break out of current loop and insert next one
            
    if (yStart+1==maxY): 
        print(f'Structures reached Y limit after only {i} particles')
        break

display(A)
        

i= 0 	x=117 	y=1 	yStart=6
i= 1000 	x=271 	y=14 	yStart=27
i= 2000 	x=179 	y=11 	yStart=39
i= 3000 	x=320 	y=41 	yStart=57
i= 4000 	x=29 	y=68 	yStart=73
i= 5000 	x=269 	y=64 	yStart=87
i= 6000 	x=131 	y=68 	yStart=103
i= 7000 	x=184 	y=92 	yStart=121
i= 8000 	x=180 	y=99 	yStart=137
i= 9000 	x=42 	y=138 	yStart=155
i= 10000 	x=194 	y=126 	yStart=169
i= 11000 	x=422 	y=155 	yStart=181
i= 12000 	x=178 	y=185 	yStart=193
i= 13000 	x=278 	y=184 	yStart=209
i= 14000 	x=425 	y=201 	yStart=224
i= 15000 	x=19 	y=234 	yStart=239
Structures reached Y limit after only 15538 particles


In [10]:
def run_dla(nParticles=10000, maxX=500, maxY=250, pLeft=0.25, pRight=0.25, p=1.0):
    A = np.zeros((maxX, maxY))
    A[:,0] = 1 # sticky wall at the bottom
    yBuffer = 5
    yStart  = 1 + yBuffer

    for i in range(0, nParticles):
        x = np.random.randint(0, maxX)
        y = yStart

        while True:
            xOrg = x
            yOrg = y

            r = np.random.random()
            if r < pLeft:# left
                x = x - 1
            elif r < pLeft + pRight: # right
                x = x + 1
            elif r < pLeft + pRight + 0.25: # up
                y = y + 1
            else: # down
                y = y - 1

            # periodic boundary conditions in x
            if x == -1:
                x = maxX - 1
            if x == maxX:
                x = 0

            if (A[x,y] == 1 or y > yStart):
                x = xOrg
                y = yOrg
                continue

            xp = x + 1
            xm = x - 1
            yp = y + 1
            ym = y - 1
            if xm == -1:
                xm = maxX - 1
            if xp == maxX:
                xp = 0

            neighbor_is_filled = False
            if A[xp,y] == 1 or A[xm,y] == 1 or A[x,yp] == 1 or A[x,ym] == 1:
                neighbor_is_filled = True
            #for part 3:
            if neighbor_is_filled:
                #  only stick with probability p, otherwise keep diffusing
                if np.random.random() < p:
                    A[x,y] = 1
                    if (y+yBuffer > yStart and y+yBuffer < maxY):
                        yStart = y + yBuffer
                    break

        if (yStart+1 == maxY):
            print(f'Structures reached Y limit after only {i} particles')
            break

    return A

def show(A, title, nRows=None):
    if nRows is None:
        nRows = A.shape[1]
    plt.figure(figsize=(10, 10*nRows/A.shape[0] + 0.5))
    plt.imshow(A[:, :nRows].T, origin='lower', interpolation='nearest')
    plt.title(title)
    plt.axis('off')
    plt.show()

def height(A):
    return np.max(np.nonzero(A.any(axis=0))[0])

In [11]:
np.random.seed(1)
A2 = run_dla(nParticles=10000, maxX=500, maxY=250, pLeft=0.10, pRight=0.40)
show(A2, 'Part II: flow to the right  (P_right=0.40, P_left=0.10)')
plt.imsave('part2_flow.png', A2.T, origin='lower')

Structures reached Y limit after only 4132 particles


### Interpretation of Results
The dendrites grow against the stream like stalagmites! the branches lean to the left and upstream into the flow. This is because the particles drift to the right, so they mostly approach the aggregate from the left. The face of the branches facing upstream catch almost all incoming particles. Each new particle attaches on the upstream side, so the tip keeps moving in that direction. 

In [12]:
pValues = [1.0, 0.3, 0.1, 0.03, 0.01]
results = {}
for p in pValues:
    np.random.seed(1)
    A3 = run_dla(nParticles=10000, maxX=500, maxY=250, p=p)
    results[p] = A3
    h = height(A3)
    density = A3[:,1:].sum() / (A3.shape[0]*h)
    print(f'p = {p:5}:  height = {h:3d} rows,  filling fraction = {density:.2f}')
    show(A3, f'Part III: p = {p}   (height {h}, filling fraction {density:.2f})', nRows=180)
    plt.imsave(f'part3_p{p}.png', A3[:, :180].T, origin='lower')

p =   1.0:  height = 161 rows,  filling fraction = 0.12
p =   0.3:  height = 127 rows,  filling fraction = 0.16
p =   0.1:  height =  98 rows,  filling fraction = 0.20
p =  0.03:  height =  50 rows,  filling fraction = 0.40
p =  0.01:  height =  44 rows,  filling fraction = 0.45
